In [ ]:
# STEP 0 - run FIRST in a fresh kernel. Stops early (with advice) if the GPU is not healthy.
import os, subprocess, time
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"      # DEBUG ONLY: uncomment BEFORE importing torch to find the exact failing line (slower)
import torch
def gpu_health():
    if not torch.cuda.is_available(): raise SystemExit("CUDA is not available in this kernel. Select the 'Python (GPU)' kernel.")
    try:
        a = torch.randn(2048, 2048, device="cuda"); b = (a @ a).sum(); torch.cuda.synchronize()
        x = torch.randn(64, 3, 224, 224, device="cuda"); y = torch.nn.functional.conv2d(x, torch.randn(8, 3, 3, 3, device="cuda")); torch.cuda.synchronize()
        free, total = torch.cuda.mem_get_info(); print(f"GPU OK: {torch.cuda.get_device_name(0)} | free {free/1e9:.1f} of {total/1e9:.1f} GB")
        del a, b, x, y; torch.cuda.empty_cache(); return True
    except Exception as e:
        print("GPU TEST FAILED:", type(e).__name__, str(e)[:200])
        print("\n=== What to do ===")
        print("1) Kernel > Restart Kernel (a CUDA 'illegal memory access' is sticky: every later CUDA call in this kernel fails until you restart).")
        print("2) Close every other notebook/kernel that uses the GPU, then run 'nvidia-smi' in a terminal and check no python process is holding memory.")
        print("3) If it fails again in a FRESH kernel: reboot Windows, update the NVIDIA driver, and reset any GPU overclock/undervolt tool (for example NZXT CAM profiles) to stock.")
        return False
try: print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,temperature.gpu", "--format=csv"], capture_output=True, text=True, timeout=20).stdout)
except Exception: pass
assert gpu_health(), "GPU not healthy: follow the advice printed above"
dev = "cuda"

In [ ]:
# STEP 1 - setup (needs Step 0 first)
import gc, re, json, shutil, datetime
import numpy as np, pandas as pd, timm
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from scipy.optimize import minimize_scalar
from sklearn.metrics import average_precision_score, roc_curve
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm

VB  = r"G:\VinBig\vinbigdata"                      # your folder: train/, test/, train.csv, test.csv
OUT = r"G:\runs"; DG = rf"{OUT}\dg"; XD = rf"{OUT}\part34"
WD  = rf"{OUT}\vinbig"; os.makedirs(WD, exist_ok=True)   # everything this notebook creates goes here
EPOCHS, SEEDS, L = 7, [0, 1, 2], 6
LABELS6 = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]   # model output order
FIND5 = ["Cardiomegaly", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]             # Edema does not exist in VinBigData
IDX5 = [LABELS6.index(l) for l in FIND5]
SCALE = "minmax"     # DICOM to 8-bit: "minmax" or "percentile"; change only if Step 5 shows a contrast mismatch

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(rf"{WD}\{name}.pdf", bbox_inches="tight"); fig.savefig(rf"{WD}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()
print("device:", dev, "| outputs go to:", WD)

In [ ]:
# STEP 2 - look at the folders
FILES = {}
for sub in ["train", "test"]:
    FILES[sub] = {os.path.splitext(e.name)[0]: e.path for e in os.scandir(rf"{VB}\{sub}") if e.is_file()}
    ext = pd.Series([os.path.splitext(p)[1].lower() for p in FILES[sub].values()]).value_counts().to_dict()
    print(f"{sub}: {len(FILES[sub])} files | extensions {ext} | example: {list(FILES[sub].values())[:1]}")
assert len(FILES["train"]) > 0, "no image files found in the train folder (are they inside sub-folders?)"

In [ ]:
# STEP 3 - image-level labels from train.csv (main rule: >= 2 of 3 radiologists)
d = pd.read_csv(rf"{VB}\train.csv")
ids_all = d.image_id.unique()
pos = d[d.class_name != "No finding"].drop_duplicates(["image_id", "rad_id", "class_name"])
cnt = (pos.groupby(["image_id", "class_name"]).rad_id.nunique().unstack(fill_value=0)
          .reindex(ids_all, fill_value=0).rename_axis("image_id"))
MAP = {"Cardiomegaly": "Cardiomegaly", "Consolidation": "Consolidation", "Atelectasis": "Atelectasis",
       "Pneumothorax": "Pneumothorax", "Pleural Effusion": "Pleural effusion"}
lab = pd.DataFrame(index=cnt.index)
for name, col in MAP.items():
    n = cnt[col]
    lab[name] = (n >= 2).astype(int)            # main rule: at least 2 of the 3 radiologists
    lab[name + "_any"] = (n >= 1).astype(int)   # sensitivity rule: any radiologist
    lab[name + "_n"] = n                        # number of agreeing radiologists (0 to 3)
lab = lab.reset_index()
found = lab.image_id.isin(FILES["train"]); print("labelled images:", len(lab), "| image file found for:", int(found.sum()))
lab = lab[found].reset_index(drop=True); lab.to_csv(rf"{WD}\vinbig_image_labels.csv", index=False)
tab = pd.DataFrame({"majority >=2 of 3 (main)": lab[FIND5].sum(), "any radiologist": lab[[f + "_any" for f in FIND5]].sum().values,
                    "unanimous (3 of 3)": [(lab[f + "_n"] == 3).sum() for f in FIND5]}); tab["prevalence % (main)"] = (100 * lab[FIND5].mean()).round(2)
display(tab)

In [ ]:
# STEP 4 - build the 224 px cache (resumable; if it is already finished this just opens it)
def read_gray8(path):
    """any image file -> 8-bit grayscale at full resolution, bright bones / dark air"""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".dicom", ".dcm"):
        import pydicom
        try: from pydicom.pixels import apply_voi_lut
        except ImportError: from pydicom.pixel_data_handlers.util import apply_voi_lut
        ds = pydicom.dcmread(path); a = ds.pixel_array
        try: a = apply_voi_lut(a, ds)
        except Exception: pass
        a = a.astype(np.float32)
        if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1": a = a.max() - a      # inverted DICOMs
    else:
        im = Image.open(path)
        if im.mode in ("I;16", "I", "F"): a = np.asarray(im).astype(np.float32)
        else: return np.asarray(im.convert("L"))                                                # 8-bit png/jpg: same as your NIH/CheXpert caches
    lo, hi = (a.min(), a.max()) if SCALE == "minmax" else np.percentile(a, [0.5, 99.5])
    return np.clip((a - lo) / max(hi - lo, 1e-6) * 255, 0, 255).astype(np.uint8)

def to224(a8):                                       # identical to your NIH and CheXpert caches: direct resize, bilinear
    return np.asarray(Image.fromarray(a8).resize((224, 224), Image.BILINEAR))

CACHE, PROG = rf"{WD}\vb_train_224.u8", rf"{WD}\vb_train_progress.json"
ids = lab.image_id.values; files = [FILES["train"][i] for i in ids]; N = len(ids)
state = json.load(open(PROG)) if os.path.exists(PROG) else dict(next=0, bad=[])
if state["next"] < N:
    mm = np.memmap(CACHE, dtype=np.uint8, mode="r+" if os.path.exists(CACHE) else "w+", shape=(N, 224, 224))
    def put(i):
        try: mm[i] = to224(read_gray8(files[i]))
        except Exception as e: state["bad"].append(int(i)); print("unreadable:", files[i], type(e).__name__, str(e)[:80])
    for s0 in tqdm(range(state["next"], N, 1000), desc="caching VinBigData at 224 px (resumable)"):
        with ThreadPoolExecutor(8) as ex: list(ex.map(put, range(s0, min(s0 + 1000, N))))
        mm.flush(); state["next"] = min(s0 + 1000, N); json.dump(state, open(PROG, "w"))
    del mm
VB_A = np.memmap(CACHE, dtype=np.uint8, mode="r", shape=(N, 224, 224))
VALID = np.ones(N, bool); VALID[state["bad"]] = False; CI = np.where(VALID)[0]       # CI: cache index of each usable image
LAB = lab[VALID].reset_index(drop=True)                                               # labels aligned with every array below
Y5, Yany, Ncnt = LAB[FIND5].values, LAB[[f + "_any" for f in FIND5]].values, LAB[[f + "_n" for f in FIND5]].values
print("cache ready:", VB_A.shape, "| unreadable:", len(state["bad"]), "| usable images:", len(LAB))

In [ ]:
# STEP 5a - load the other two sites (needed again after every kernel restart; no figures here)
nih = pd.read_csv(r"G:\nih_split.csv"); te_i = np.where(nih.official.values == "test")[0]
NIH_A = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224)); CX_T = np.load(r"G:\cx_test_224.npy")
print("NIH test images:", len(te_i), "| CheXpert test images:", len(CX_T))

In [ ]:
# STEP 5b - preprocessing check (do not skip): does VinBigData look like your other two sites?
rng = np.random.default_rng(0)
SITES = {"NIH": NIH_A[np.sort(rng.choice(te_i, 1000, replace=False))],
         "CheXpert": CX_T[np.sort(rng.choice(len(CX_T), 1000, replace=False))],
         "VinBigData": VB_A[np.sort(rng.choice(CI, 1000, replace=False))]}
rows = []
for k, x in SITES.items():
    f = x.reshape(len(x), -1).astype(np.float32)
    rows.append({"site": k, "mean intensity": f.mean(), "contrast (std)": f.std(1).mean(),
                 "% pixels <5": 100 * (f < 5).mean(), "% pixels >250": 100 * (f > 250).mean()})
stats = pd.DataFrame(rows).set_index("site").round(2); display(stats); stats.to_csv(rf"{WD}\table_preprocessing_check.csv")

fig, ax = plt.subplots(3, 6, figsize=(COL2, 3.8))
for r_, (k, x) in enumerate(SITES.items()):
    for c_ in range(6): ax[r_, c_].imshow(x[c_ * 150], cmap="gray", vmin=0, vmax=255); ax[r_, c_].axis("off")
    ax[r_, 0].set_title(k, loc="left", fontsize=7)
fig.tight_layout(); save_fig(fig, "fig_vinbig_samples")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.4))
for k, x in SITES.items():
    f = x.reshape(len(x), -1).astype(np.float32)
    ax[0].hist(f.mean(1), bins=40, alpha=0.5, density=True, label=k); ax[1].hist(f.std(1), bins=40, alpha=0.5, density=True, label=k)
ax[0].set(xlabel="Per-image mean intensity", ylabel="Density"); ax[1].set(xlabel="Per-image contrast (std)"); ax[0].legend(frameon=False)
fig.tight_layout(); save_fig(fig, "fig_intensity_distributions")
print("CHECK: bones should look bright and the lungs dark in every row. If VinBigData is inverted, tell me before going on.")

In [ ]:
# STEP 6 - predictions of all 9 model families (hardened: health check, small batches, errors name the model, finished models are skipped)
assert gpu_health(), "GPU not healthy: restart the kernel and rerun Steps 0, 1, 2, 3, 4, 5a first"
DGF = ["ERM (natural)", "ERM (balanced)", "DANN", "GroupDRO", "CORAL"]; DGT = {"ERM (natural)": "erm_nat", "ERM (balanced)": "erm_bal", "DANN": "dann", "GroupDRO": "gdro", "CORAL": "coral"}
SINGLE = ["DenseNet NIH-only", "DenseNet CheXpert-only", "ConvNeXt NIH-only", "ConvNeXt CheXpert-only"]
ARCH = {f: "densenet121" for f in DGF + SINGLE[:2]}; ARCH.update({f: "convnext_tiny.fb_in1k" for f in SINGLE[2:]})
BASE = {f: (lambda s, t=DGT[f]: rf"{DG}\{t}_e{EPOCHS}_seed{s}") for f in DGF}
BASE["DenseNet NIH-only"]      = lambda s: rf"{OUT}\nih_train\densenet121_nih_e{EPOCHS}_seed{s}"
BASE["DenseNet CheXpert-only"] = lambda s: rf"{OUT}\densenet121_e{EPOCHS}_seed{s}"
BASE["ConvNeXt NIH-only"]      = lambda s: rf"{XD}\convnext_tiny_nih_e{EPOCHS}_seed{s}"
BASE["ConvNeXt CheXpert-only"] = lambda s: rf"{XD}\convnext_tiny_cx_e{EPOCHS}_seed{s}"
SRCP = {f: (lambda s, f=f: BASE[f](s) + "_preds.npz") for f in DGF + ["DenseNet NIH-only", "ConvNeXt NIH-only", "ConvNeXt CheXpert-only"]}
SRCP["DenseNet CheXpert-only"] = lambda s: rf"{OUT}\densenet121_e{EPOCHS}_seed{s}_results\preds.npz"
FAMS = DGF + SINGLE; POOLED = set(DGF)
miss = [p for f in FAMS for s in SEEDS for p in (BASE[f](s) + ".pt", SRCP[f](s)) if not os.path.exists(p)]
print("missing files:", len(miss)); [print("  ", p) for p in miss]; assert not miss, "fix the paths above first"

MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
def prep(A, idx):
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - MEAN) / STD
@torch.inference_mode()
def predict_logits(m, A, bs=128):
    m.eval(); out = []
    for i in range(0, len(A), bs):
        with torch.autocast(dev, enabled=(dev == "cuda")): o = m(prep(A, range(i, min(i + bs, len(A)))))
        out.append(o.float().cpu())
    torch.cuda.synchronize(); return torch.cat(out).numpy()
slug = lambda f: re.sub(r"[^A-Za-z0-9]+", "_", f)
def load_model(fam, s):
    m = timm.create_model(ARCH[fam], pretrained=False, num_classes=L).to(dev)
    m.load_state_dict(torch.load(BASE[fam](s) + ".pt", map_location=dev, weights_only=True)); return m.eval()

for fam in FAMS:
    for s in SEEDS:
        f = rf"{WD}\logit_{slug(fam)}_seed{s}.npy"
        if os.path.exists(f): continue
        try:
            m = load_model(fam, s); z = predict_logits(m, VB_A); assert np.isfinite(z).all(), "non-finite logits"
            np.save(f, z); del m; gc.collect(); torch.cuda.empty_cache(); print("predicted:", fam, "seed", s)
        except Exception as e:
            print(f"\nFAILED on {fam}, seed {s}: {type(e).__name__}: {str(e)[:160]}")
            if "illegal memory" in str(e) or "CUDA error" in str(e): print("CUDA context is now broken: Kernel > Restart, then rerun Steps 0, 1, 2, 3, 4, 5a and this step (finished models are skipped).")
            raise

sig = lambda z: 1 / (1 + np.exp(-z))
LG = {f: {s: np.load(rf"{WD}\logit_{slug(f)}_seed{s}.npy")[VALID] for s in SEEDS} for f in FAMS}   # logits of all 6 outputs
PR5 = {f: {s: sig(LG[f][s])[:, IDX5] for s in SEEDS} for f in FAMS}                                  # probabilities of the 5 findings
print("predictions ready for", len(FAMS), "model families x", len(SEEDS), "seeds on", len(LAB), "VinBigData images")

In [ ]:
# STEP 7 - accuracy tables (thresholds always come from the SOURCE validation sets, never from VinBigData)
from scipy.stats import rankdata
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def ece_of(y, p, nb=10):
    bi = np.clip((p * nb).astype(int), 0, nb - 1); e = 0.0
    for k in range(nb):
        mk = bi == k
        if mk.any(): e += mk.mean() * abs(p[mk].mean() - y[mk].mean())
    return e
GRID = np.linspace(0.02, 0.95, 94)
def best_thr(y, p):
    pr = p[:, None] >= GRID[None, :]; y1 = (y == 1)[:, None]; tp = (pr & y1).sum(0); fp = pr.sum(0) - tp; fn = y1.sum() - tp
    return GRID[np.argmax(2 * tp / np.maximum(2 * tp + fp + fn, 1))]
def f1_at(y, p, t):
    pr = p >= t; y1 = y == 1; tp = (pr & y1).sum(); dn = 2 * tp + (pr & ~y1).sum() + (~pr & y1).sum()
    return 2 * tp / dn if dn else 0.0

def source_info(fam, s):
    z = np.load(SRCP[fam](s))
    if fam in POOLED:                      return [(z["nih_val_Y"], z["nih_val_P"]), (z["cx_val_Y"], z["cx_val_P"])], [(z["nih_test_Y"], z["nih_test_P"]), (z["cx_test_Y"], z["cx_test_P"])]
    if fam == "DenseNet CheXpert-only":    return [(z["val_Y"], z["val_P"])], [(z["test_Y"], z["test_P"])]
    if fam == "DenseNet NIH-only":         return [(z["nih_val_Y"], z["nih_val_P"])], [(z["nih_test_Y"], z["nih_test_P"])]
    if fam == "ConvNeXt NIH-only":         return [(z["val_Y"], z["val_P"])], [(z["nih_test_Y"], z["nih_test_P"])]
    return [(z["val_Y"], z["val_P"])], [(z["cx_test_Y"], z["cx_test_P"])]
cat5 = lambda pairs, k: np.concatenate([np.asarray(p[k])[:, IDX5] for p in pairs])

rows, SEEN, THR = [], {}, {}
for fam in FAMS:
    for s in SEEDS:
        val, seen = source_info(fam, s)
        Yv, Pv = cat5([(y.astype(int), p) for y, p in val], 0), cat5(val, 1)
        thr = np.array([best_thr(Yv[:, j], Pv[:, j]) for j in range(5)]); THR[(fam, s)] = thr
        SEEN[(fam, s)] = np.mean([np.mean([fauc(np.asarray(y)[:, k].astype(int), np.asarray(p)[:, k]) for k in IDX5]) for y, p in seen])
        P = PR5[fam][s]
        for j, f in enumerate(FIND5):
            rows.append(dict(fam=fam, seed=s, finding=f, auroc=fauc(Y5[:, j], P[:, j]), auprc=average_precision_score(Y5[:, j], P[:, j]),
                             ece=ece_of(Y5[:, j], P[:, j]), f1=f1_at(Y5[:, j], P[:, j], thr[j]), acc=float(((P[:, j] >= thr[j]) == (Y5[:, j] == 1)).mean())))
M = pd.DataFrame(rows); M.to_csv(rf"{WD}\metrics_per_seed_per_finding.csv", index=False)
S = M.groupby(["fam", "seed"])[["auroc", "auprc", "ece", "f1", "acc"]].mean().reset_index()
S["seen"] = [SEEN[(f, s)] for f, s in zip(S.fam, S.seed)]; S["gap"] = S.seen - S.auroc
pm = lambda x: f"{x.mean():.3f} ± {x.std(ddof=1):.3f}"
tabA = S.groupby("fam")[["auroc", "auprc", "ece", "f1", "acc", "seen", "gap"]].agg(pm).loc[FAMS]
tabA.columns = ["AUROC on VinBigData", "AUPRC", "ECE", "F1", "Accuracy", "AUROC on seen sites", "Gap (seen - VinBig)"]
display(tabA); tabA.to_csv(rf"{WD}\table_vinbig_main.csv")
perf = M.groupby(["fam", "finding"]).auroc.mean().unstack()[FIND5].loc[FAMS]; display(perf.round(3)); perf.to_csv(rf"{WD}\table_vinbig_per_finding_auroc.csv")

# sensitivity to the label rule (AUROC, mean over seeds)
def auc_rule(Yr, P, mk, j):
    if mk is None: return fauc(Yr[:, j], P[:, j])
    k = mk[:, j]; return fauc(Yr[k, j], P[k, j])
RULES = {"majority >=2 of 3 (main)": (Y5, None), "any radiologist": (Yany, None), "exactly-1-rater images removed": (Y5, Ncnt != 1)}
sens = pd.DataFrame({r: {f: np.mean([np.mean([auc_rule(Yr, PR5[f][s], mk, j) for j in range(5)]) for s in SEEDS]) for f in FAMS} for r, (Yr, mk) in RULES.items()})
display(sens.round(3)); sens.to_csv(rf"{WD}\table_label_rule_sensitivity.csv")

In [ ]:
# STEP 8 - image-level bootstrap (each VinBigData image is a different person), paired across all models
B = 300; rng = np.random.default_rng(0); N5 = len(Y5)
PT5 = {f: np.array([np.mean([fauc(Y5[:, j], PR5[f][s][:, j]) for s in SEEDS]) for j in range(5)]) for f in FAMS}
BS_ = {f: np.zeros((B, 5)) for f in FAMS}
for b in tqdm(range(B), desc="bootstrap"):
    ix = rng.integers(0, N5, N5)
    for f in FAMS: BS_[f][b] = [np.mean([fauc(Y5[ix, j], PR5[f][s][ix, j]) for s in SEEDS]) for j in range(5)]
ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)

rows = []
for f in FAMS:
    lo, hi = ci(BS_[f].mean(1)); rows.append(dict(model=f, mean_AUROC=PT5[f].mean(), CI=f"[{lo:.3f}, {hi:.3f}]",
                **{ff: f"{PT5[f][j]:.3f} [{ci(BS_[f][:, j])[0]:.3f}, {ci(BS_[f][:, j])[1]:.3f}]" for j, ff in enumerate(FIND5)}))
CIT = pd.DataFrame(rows).set_index("model"); display(CIT); CIT.to_csv(rf"{WD}\table_vinbig_auroc_with_CI.csv")

best_single = max(SINGLE, key=lambda f: PT5[f].mean()); print("best single-site model:", best_single)
rows = []
for base in ["ERM (natural)", "ERM (balanced)", best_single]:
    for f in FAMS:
        if f == base: continue
        dlt = BS_[f].mean(1) - BS_[base].mean(1); lo, hi = np.percentile(dlt, [2.5, 97.5])
        rows.append(dict(reference=base, model=f, mean_diff=round(float(dlt.mean()), 4), CI=f"[{lo:+.4f}, {hi:+.4f}]", excludes_zero="yes" if (lo > 0 or hi < 0) else "no"))
CMP = pd.DataFrame(rows); display(CMP); CMP.to_csv(rf"{WD}\table_vinbig_paired_differences.csv", index=False)

In [ ]:
# STEP 9 - figures
HAT = lambda f: "" if f in POOLED else "//"; FC = lambda f: "0.35" if f in POOLED else "0.8"
# 9A: mean AUROC with bootstrap CI
pt = {f: PT5[f].mean() for f in FAMS}; cis = {f: np.percentile(BS_[f].mean(1), [2.5, 97.5]) for f in FAMS}
fig, ax = plt.subplots(figsize=(COL2, 3.0))
for i, f in enumerate(FAMS):
    ax.bar(i, pt[f], yerr=[[pt[f] - cis[f][0]], [cis[f][1] - pt[f]]], capsize=2, color=FC(f), edgecolor="k", lw=0.4, hatch=HAT(f))
ax.set_xticks(range(len(FAMS))); ax.set_xticklabels(FAMS, rotation=30, ha="right"); ax.set_ylim(min(pt.values()) - 0.06, max(pt.values()) + 0.04)
ax.set_ylabel("AUROC (VinBigData, 5 findings)"); ax.grid(True, axis="y", alpha=0.3)
ax.bar(0, 0, color="0.35", label="trained on NIH + CheXpert"); ax.bar(0, 0, color="0.8", hatch="//", edgecolor="k", label="trained on one hospital"); ax.legend(frameon=False)
fig.tight_layout(); save_fig(fig, "fig_vinbig_auroc")

# 9B: per-finding heatmaps (absolute and difference to ERM natural)
H = pd.DataFrame({f: PT5[f] for f in FAMS}, index=FIND5).T
fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.4))
for a_, (D, ttl, cm) in zip(ax, [(H, "AUROC", "viridis"), (H - H.loc["ERM (natural)"], "Difference to ERM (natural)", "coolwarm")]):
    v = np.abs(D.values).max() if cm == "coolwarm" else None
    im = a_.imshow(D.values, cmap=cm, aspect="auto", **({"vmin": -v, "vmax": v} if v else {}))
    for i in range(D.shape[0]):
        for j in range(D.shape[1]): a_.text(j, i, f"{D.values[i, j]:.3f}" if cm == "viridis" else f"{D.values[i, j]:+.3f}", ha="center", va="center", fontsize=5.5,
                                            color=("w" if D.values[i, j] < (D.values.min() + D.values.max()) / 2 else "k") if cm == "viridis" else "k")
    a_.set_xticks(range(5)); a_.set_xticklabels(FIND5, rotation=40, ha="right"); a_.set_yticks(range(len(FAMS))); a_.set_yticklabels(FAMS if a_ is ax[0] else []); a_.set_title(ttl); a_.grid(False)
fig.tight_layout(); save_fig(fig, "fig_vinbig_per_finding")

# 9C: ROC curves (seed 0)
REP = ["ERM (natural)", "ERM (balanced)", "DANN", "DenseNet NIH-only", "DenseNet CheXpert-only"]
fig, axs = plt.subplots(2, 3, figsize=(COL2, 4.4)); axs = axs.ravel()
for j, f in enumerate(FIND5):
    for k, fam in enumerate(REP):
        fpr, tpr, _ = roc_curve(Y5[:, j], PR5[fam][0][:, j]); axs[j].plot(fpr, tpr, color=f"C{k}", label=fam)
    axs[j].plot([0, 1], [0, 1], "k:", lw=0.6); axs[j].set_title(f"{f} ({int(Y5[:, j].sum())} positives)"); axs[j].set(xlabel="False positive rate", ylabel="True positive rate"); axs[j].grid(True, alpha=0.3)
h, l_ = axs[0].get_legend_handles_labels(); axs[5].axis("off"); axs[5].legend(h, l_, loc="center", frameon=False)
fig.tight_layout(); save_fig(fig, "fig_vinbig_roc")

# 9D: reliability (all five findings pooled, equal-count bins, log axes)
def reliab_q(y, p, nb=10):
    o = np.argsort(p); bins = np.array_split(o, nb); return np.array([p[b].mean() for b in bins]), np.array([y[b].mean() for b in bins])
fig, ax = plt.subplots(figsize=(COL1 * 1.3, 3.2))
for k, fam in enumerate(REP):
    px, py = reliab_q(Y5.ravel(), PR5[fam][0].ravel()); ax.plot(px, np.maximum(py, 1e-4), "-o", ms=2.5, color=f"C{k}", label=fam)
ax.plot([1e-4, 1], [1e-4, 1], "k:", lw=0.6); ax.set_xscale("log"); ax.set_yscale("log"); ax.set(xlabel="Mean predicted probability", ylabel="Observed frequency"); ax.legend(frameon=False, fontsize=5.5); ax.grid(True, alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_vinbig_reliability")

# 9E: seen sites versus VinBigData
sa = S.groupby("fam")[["seen", "auroc"]].agg(["mean", "std"]).loc[FAMS]
fig, ax = plt.subplots(figsize=(COL2, 3.0)); w = 0.38; xs = np.arange(len(FAMS))
ax.bar(xs - w / 2, sa[("seen", "mean")], w, yerr=sa[("seen", "std")], capsize=1.5, color="0.35", edgecolor="k", lw=0.4, label="Seen sites (NIH / CheXpert tests)")
ax.bar(xs + w / 2, sa[("auroc", "mean")], w, yerr=sa[("auroc", "std")], capsize=1.5, color="0.8", edgecolor="k", lw=0.4, label="Unseen site (VinBigData)")
ax.set_xticks(xs); ax.set_xticklabels(FAMS, rotation=30, ha="right"); ax.set_ylim(min(sa[("auroc", "mean")]) - 0.06, None); ax.set_ylabel("AUROC (same 5 findings)"); ax.legend(frameon=False); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_seen_vs_unseen")

# 9F: calibration and thresholded metrics
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.7))
for a_, (col, ttl) in zip(ax, [("ece", "ECE (lower is better)"), ("f1", "Macro-F1 at source thresholds"), ("auprc", "AUPRC")]):
    g = S.groupby("fam")[col].agg(["mean", "std"]).loc[FAMS]
    for i, f in enumerate(FAMS): a_.bar(i, g.loc[f, "mean"], yerr=g.loc[f, "std"], capsize=1.5, color=FC(f), edgecolor="k", lw=0.4, hatch=HAT(f))
    a_.set_xticks(range(len(FAMS))); a_.set_xticklabels(FAMS, rotation=60, ha="right", fontsize=5.5); a_.set_title(ttl); a_.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_vinbig_ece_f1_auprc")

In [ ]:
# STEP 10 - domain representation (needs Step 5a)
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
NS = 1000; rng = np.random.default_rng(1)
IDXS = {"NIH": (NIH_A, np.sort(rng.choice(te_i, NS, replace=False))), "CheXpert": (CX_T, np.sort(rng.choice(len(CX_T), NS, replace=False))),
        "VinBigData": (VB_A, np.sort(rng.choice(CI, NS, replace=False)))}
@torch.inference_mode()
def feats(m, A, idx, bs=125):
    out = []
    for i in range(0, len(idx), bs):
        with torch.autocast(dev, enabled=(dev == "cuda")): f = m.forward_head(m.forward_features(prep(A, idx[i:i + bs])), pre_logits=True)
        out.append(f.float().cpu())
    return torch.cat(out).numpy()
clf = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=300))
FE, rows = {}, []
for fam in ["ERM (natural)", "DANN"]:
    m = load_model(fam, 0); FE[fam] = {k: feats(m, A, ix) for k, (A, ix) in IDXS.items()}; del m; gc.collect(); torch.cuda.empty_cache()
    pairs = {"NIH vs CheXpert": ["NIH", "CheXpert"], "VinBigData vs NIH": ["VinBigData", "NIH"], "VinBigData vs CheXpert": ["VinBigData", "CheXpert"], "3 sites (chance 0.33)": ["NIH", "CheXpert", "VinBigData"]}
    for nm, ks in pairs.items():
        X = np.concatenate([FE[fam][k] for k in ks]); y = np.repeat(np.arange(len(ks)), NS)
        rows.append({"model": fam, "task": nm, "site-classifier accuracy": cross_val_score(clf(), X, y, cv=5).mean()})
DOM = pd.DataFrame(rows).pivot(index="task", columns="model", values="site-classifier accuracy").round(3); display(DOM); DOM.to_csv(rf"{WD}\table_domain_classifier.csv")

fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.2)); colr = {"NIH": "C0", "CheXpert": "C1", "VinBigData": "C3"}
for a_, fam in zip(ax, ["ERM (natural)", "DANN"]):
    X = np.concatenate([FE[fam][k] for k in colr]); Z = TSNE(perplexity=30, init="pca", random_state=0).fit_transform(PCA(50, random_state=0).fit_transform(StandardScaler().fit_transform(X)))
    for i, k in enumerate(colr): a_.scatter(Z[i * NS:(i + 1) * NS, 0], Z[i * NS:(i + 1) * NS, 1], s=3, alpha=0.5, color=colr[k], label=k)
    a_.set_title(f"t-SNE of {fam} features"); a_.set_xticks([]); a_.set_yticks([])
ax[0].legend(frameon=False, markerscale=3); fig.tight_layout(); save_fig(fig, "fig_domain_tsne")

In [ ]:
# STEP 11 - recalibration with k labelled VinBigData images (split 50/50 by image into a pool half and an evaluation half)
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def fit_bias(z, y): return minimize_scalar(lambda b: np.mean(np.logaddexp(0, z + b) - y * (z + b)), bounds=(-8, 8), method="bounded").x
def fit_platt(z, y):
    lr = LogisticRegression(C=1e6, max_iter=500).fit(z.reshape(-1, 1), y); return float(np.clip(lr.coef_[0, 0], 0.1, 5.0)), float(lr.intercept_[0])
RMETH = ["Threshold only", "Bias (intercept)", "Platt (slope+intercept)"]
def apply_method(meth, zp, yp, ze, thr0):
    a, b, t, fit = 1.0, 0.0, thr0, False; npos = int(yp.sum()); nneg = len(yp) - npos
    if meth == "Threshold only":
        if npos >= 3 and nneg >= 3: t = best_thr(yp, sig(zp)); fit = True
    elif npos >= 5 and nneg >= 5:
        if meth.startswith("Bias"): b = fit_bias(zp, yp)
        else: a, b = fit_platt(zp, yp)
        t = best_thr(yp, sig(a * zp + b)); fit = True
    return sig(a * ze + b), t, fit
KS, DRAWS, RFAMS = [0, 50, 100, 250, 500, 1000, 2500, 5000], 10, ["ERM (natural)", "DenseNet NIH-only", "DenseNet CheXpert-only"]
rows = []
for fam in RFAMS:
    for s in SEEDS:
        Z = LG[fam][s][:, IDX5]; thr0 = THR[(fam, s)]
        for dr in range(DRAWS):
            perm = np.random.default_rng([s, dr]).permutation(len(Y5)); pool, ev = perm[:len(perm) // 2], perm[len(perm) // 2:]
            for K in KS:
                if K == 0 and dr > 0: continue
                ip = pool[:K]
                for meth in RMETH:
                    es, fs, nf = [], [], 0
                    for j in range(5):
                        pe, t, fit = apply_method(meth, Z[ip, j], Y5[ip, j], Z[ev, j], thr0[j])
                        es.append(ece_of(Y5[ev, j], pe)); fs.append(f1_at(Y5[ev, j], pe, t)); nf += fit
                    rows.append(dict(fam=fam, seed=s, method=meth, K=K, draw=dr, ece=np.mean(es), f1=np.mean(fs), n_fit=nf))
RC = pd.DataFrame(rows); RC.to_csv(rf"{WD}\recalibration_raw.csv", index=False)
ps = RC.groupby(["fam", "method", "K", "seed"])[["ece", "f1", "n_fit"]].mean().reset_index()
for met in ["ece", "f1"]:
    t = ps[ps.K.isin([0, 100, 500, 2500])].groupby(["fam", "method", "K"])[met].agg(pm).unstack("K"); print(f"\n{met.upper()} on the evaluation half (mean ± std over seeds)"); display(t); t.to_csv(rf"{WD}\table_vinbig_recal_{met}.csv")
print("\nfindings actually fitted (of 5) with Platt:"); display(ps[ps.method.str.startswith("Platt")].groupby(["fam", "K"]).n_fit.mean().unstack("K").round(1))

MC = {"Threshold only": "C0", "Bias (intercept)": "C2", "Platt (slope+intercept)": "C3"}
fig, axs = plt.subplots(2, 2, figsize=(COL2, 5.0))
for c_, (met, ttl) in enumerate([("ece", "ECE (lower is better)"), ("f1", "Macro-F1 (higher is better)")]):
    for m_ in RMETH:
        g = ps[(ps.fam == "ERM (natural)") & (ps.method == m_)].groupby("K")[met].agg(["mean", "std"]); axs[0, c_].plot(g.index, g["mean"], "-o", ms=2.5, color=MC[m_], label=m_)
        axs[0, c_].fill_between(g.index, g["mean"] - g["std"], g["mean"] + g["std"], color=MC[m_], alpha=0.12, lw=0)
    for i, fam in enumerate(RFAMS):
        g = ps[(ps.fam == fam) & (ps.method.str.startswith("Platt"))].groupby("K")[met].agg(["mean", "std"]); axs[1, c_].plot(g.index, g["mean"], "-o", ms=2.5, color=f"C{i}", label=fam)
        axs[1, c_].fill_between(g.index, g["mean"] - g["std"], g["mean"] + g["std"], color=f"C{i}", alpha=0.12, lw=0)
    axs[0, c_].set_title("ERM (natural): " + ttl); axs[1, c_].set_title("Platt scaling, all models: " + ttl)
    for r_ in (0, 1): axs[r_, c_].set_xscale("symlog", linthresh=50); axs[r_, c_].set_xlabel("k labelled VinBigData images"); axs[r_, c_].grid(True, alpha=0.3)
axs[0, 0].legend(frameon=False, fontsize=6); axs[1, 0].legend(frameon=False, fontsize=6); fig.tight_layout(); save_fig(fig, "fig_vinbig_recalibration")

In [ ]:
# STEP 12 - Grad-CAM on VinBigData for ERM (natural), seed 0 (hardened: small batches, empty maps counted, memory freed per finding)
import torch.nn.functional as F
assert gpu_health(), "GPU not healthy: restart the kernel and rerun Steps 0 to 6 first"
GM, GS, GN, GK = "ERM (natural)", 0, 200, 4
gm = load_model(GM, GS); P5g = PR5[GM][GS]; thr_g = THR[(GM, GS)]
for p_ in gm.parameters(): p_.requires_grad_(True)
GROUPS = ["TP", "FN", "FP", "TN"]; GNAME = {"TP": "Correct positive (TP)", "FN": "Missed positive (FN)", "FP": "False alarm (FP)", "TN": "Correct negative (TN)"}
EDGE = np.ones((224, 224), bool); EDGE[33:-33, 33:-33] = False; BOTTOM = np.zeros((224, 224), bool); BOTTOM[149:, :] = True
def gradcam(model, imgs, cls, bs=25):
    cams = []
    for i in range(0, len(imgs), bs):
        x = (torch.from_numpy(np.ascontiguousarray(imgs[i:i + bs])).to(dev).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1) - MEAN) / STD
        with torch.enable_grad():
            fm = model.forward_features(x); out = model.forward_head(fm)
            assert fm.requires_grad, "feature map has no gradient (check requires_grad)"
            g = torch.autograd.grad(out[:, cls].sum(), fm)[0]
        cam = torch.relu((g.mean((2, 3), keepdim=True) * fm).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[:, 0]
        cams.append((cam / (cam.flatten(1).max(1).values.view(-1, 1, 1) + 1e-8)).detach().cpu().numpy())
        del fm, out, g, cam, x
    torch.cuda.empty_cache(); return np.concatenate(cams)
rng = np.random.default_rng(11); rows, MEANCAM = [], {}
for j, fnd in enumerate(FIND5):
    y, p, t = Y5[:, j], P5g[:, j], thr_g[j]
    ids = {"TP": np.where((y == 1) & (p >= t))[0], "FN": np.where((y == 1) & (p < t))[0], "FP": np.where((y == 0) & (p >= t))[0], "TN": np.where((y == 0) & (p < t))[0]}
    fig, axs = plt.subplots(4, GK, figsize=(5.4, 5.6))
    for r_, g_ in enumerate(GROUPS):
        for c_ in range(GK): axs[r_, c_].axis("off")
        axs[r_, 0].text(-0.04, 0.5, f"{GNAME[g_]}\nn = {len(ids[g_])}", transform=axs[r_, 0].transAxes, rotation=90, va="center", ha="right", fontsize=5.5)
        n = min(GN, len(ids[g_]))
        if n == 0: continue
        pk = np.sort(rng.choice(ids[g_], n, replace=False)); imgs = VB_A[CI[pk]]; cams = gradcam(gm, imgs, IDX5[j])
        ok = cams.max((1, 2)) > 0.5; s_ = np.maximum(cams.sum((1, 2)), 1e-8)
        rows.append(dict(finding=fnd, group=g_, n_group=len(ids[g_]), n_used=n, empty_maps=int((~ok).sum()),
                         edge=(cams[:, EDGE].sum(1) / s_)[ok].mean() if ok.any() else np.nan, bottom=(cams[:, BOTTOM].sum(1) / s_)[ok].mean() if ok.any() else np.nan))
        if ok.any(): MEANCAM[(fnd, g_)] = cams[ok].mean(0)
        for c_ in range(min(GK, n)):
            axs[r_, c_].imshow(imgs[c_], cmap="gray"); axs[r_, c_].imshow(cams[c_], cmap="jet", alpha=0.4); axs[r_, c_].set_title(f"p = {p[pk[c_]]:.2f}", fontsize=5.5, pad=1)
    fig.suptitle(f"{GM}, trained on NIH + CheXpert, tested on VinBigData: {fnd} [threshold {t:.2f}]", fontsize=7); fig.tight_layout()
    base = rf"{WD}\gradcam_{fnd.replace(' ', '_')}"; fig.savefig(base + ".pdf", bbox_inches="tight"); fig.savefig(base + ".png", dpi=300, bbox_inches="tight")
    if fnd in ("Cardiomegaly", "Pleural Effusion"): plt.show()
    plt.close(fig); print("Grad-CAM done:", fnd)
GST = pd.DataFrame(rows); GST.to_csv(rf"{WD}\gradcam_region_stats.csv", index=False)
print("uniform-heat baselines: edge", round(EDGE.mean(), 2), "| bottom third", round(BOTTOM.mean(), 2)); display(GST.round(3))
fig, axs = plt.subplots(5, 4, figsize=(COL2, 8.4))
for r_, fnd in enumerate(FIND5):
    for c_, g_ in enumerate(GROUPS):
        axs[r_, c_].axis("off"); axs[r_, c_].imshow(MEANCAM.get((fnd, g_), np.zeros((224, 224))), cmap="jet", vmin=0, vmax=1)
        if r_ == 0: axs[r_, c_].set_title(GNAME[g_], fontsize=6)
        if c_ == 0: axs[r_, c_].text(-0.04, 0.5, fnd, transform=axs[r_, c_].transAxes, rotation=90, va="center", ha="right", fontsize=6)
fig.suptitle("Average Grad-CAM on VinBigData", fontsize=7); fig.tight_layout(); save_fig(fig, "fig_gradcam_mean_maps")
del gm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# STEP 13 - cross-hospital ensemble versus same-hospital ensembles (VinBigData is unseen by both members)
ens = lambda *Z: sig(np.mean(Z, axis=0))[:, IDX5]
nx = lambda s: (s + 1) % len(SEEDS)
CFG = {"single NIH-only":            lambda s: PR5["DenseNet NIH-only"][s],
       "single CheXpert-only":       lambda s: PR5["DenseNet CheXpert-only"][s],
       "ensemble NIH + CheXpert":    lambda s: ens(LG["DenseNet NIH-only"][s], LG["DenseNet CheXpert-only"][s]),
       "ensemble NIH + NIH (2 seeds)":      lambda s: ens(LG["DenseNet NIH-only"][s], LG["DenseNet NIH-only"][nx(s)]),
       "ensemble CheXpert + CheXpert (2 seeds)": lambda s: ens(LG["DenseNet CheXpert-only"][s], LG["DenseNet CheXpert-only"][nx(s)]),
       "pooled ERM (natural)":       lambda s: PR5["ERM (natural)"][s]}
EP = {k: {s: f(s) for s in SEEDS} for k, f in CFG.items()}
rng = np.random.default_rng(5); BE = {k: np.zeros((B, 5)) for k in CFG}
for b in tqdm(range(B), desc="bootstrap ensembles"):
    ix = rng.integers(0, N5, N5)
    for k in CFG: BE[k][b] = [np.mean([fauc(Y5[ix, j], EP[k][s][ix, j]) for s in SEEDS]) for j in range(5)]
rows = []
for k in CFG:
    pt_ = np.mean([[fauc(Y5[:, j], EP[k][s][:, j]) for j in range(5)] for s in SEEDS]); lo, hi = np.percentile(BE[k].mean(1), [2.5, 97.5])
    dlt = BE[k].mean(1) - BE["ensemble NIH + CheXpert"].mean(1); dl, dh = np.percentile(dlt, [2.5, 97.5])
    rows.append({"model": k, "AUROC": round(pt_, 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]", "diff to NIH+CheXpert ensemble": f"{dlt.mean():+.4f} [{dl:+.4f}, {dh:+.4f}]"})
ET = pd.DataFrame(rows).set_index("model"); display(ET); ET.to_csv(rf"{WD}\table_ensembles_vinbig.csv")
fig, ax = plt.subplots(figsize=(COL2, 2.8)); v = ET.AUROC.values
cl = np.array([np.percentile(BE[k].mean(1), [2.5, 97.5]) for k in CFG])
ax.bar(range(len(CFG)), v, yerr=[v - cl[:, 0], cl[:, 1] - v], capsize=2, color="0.7", edgecolor="k", lw=0.4)
ax.set_xticks(range(len(CFG))); ax.set_xticklabels(list(CFG), rotation=25, ha="right"); ax.set_ylim(v.min() - 0.05, v.max() + 0.03); ax.set_ylabel("AUROC (VinBigData, 5 findings)"); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_ensembles_vinbig")

In [ ]:
# STEP 14 - save everything into one bundle (the 224 px cache stays out of the zip)
BUN = rf"{OUT}\VINBIG_bundle"; os.makedirs(BUN, exist_ok=True)
shutil.copytree(WD, rf"{BUN}\vinbig", dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.u8", "*_last.pt", "__pycache__"))
json.dump(dict(date=str(datetime.date.today()), site="VinBigData (Kaggle release of VinDr-CXR), test-only", n_images=int(len(LAB)), findings=FIND5, label_rule="positive if >=2 of 3 radiologists marked the finding",
               sensitivity_rules=["any radiologist", "images with exactly one marking radiologist removed"], preprocessing=f"DICOM/PNG -> 8-bit gray ({SCALE}), MONOCHROME1 inverted, bilinear resize to 224x224",
               source_models=FAMS, seeds=SEEDS, thresholds="max-F1 on SOURCE validation sets only", bootstrap=B, bootstrap_unit="image",
               recalibration=dict(ks=KS, draws=DRAWS, split="50/50 by image"), gradcam=dict(model=GM, seed=GS, images_per_group=GN), torch=torch.__version__, timm=timm.__version__),
          open(rf"{BUN}\settings_vinbig.json", "w"), indent=2)
zp = shutil.make_archive(rf"{OUT}\VINBIG_bundle", "zip", BUN); print("bundle:", zp, f"({os.path.getsize(zp) / 1e6:.0f} MB)")
for e in [".csv", ".pdf", ".png", ".npy"]: print(e, sum(f.endswith(e) for _, _, fs in os.walk(BUN) for f in fs))